# Chapter 2 - Lab 5: Implementing Financial News Agent with Evaluator-Optimizer Pattern

**Definition**

**Evaluator-Optimizer** pattern is an agentic system that uses two agents:

* The first agent generates an initial response based on the user’s query.

* The second agent analyzes the response generated by the first agent and provides critical feedback. Then, it sends this back to the first agent for revision.

This system runs in a loop until a termination condition is met:

* Either the maximum number of iterations is reached,

* Or satisfactory critical feedback is obtained.

**Use Case:**

The objective of this agentic system is to design an automated agent capable of retrieving up-to-date financial news within a specified geographical region.

For implementation, the OpenAI Agents SDK framework is employed to operationalize the LLM-as-a-Judge design pattern. The retrieval process is constrained by three criteria:

* The news source must be Reuters

* Articles must be recent, i.e., published within the last two days

* The content must be relevant to a predefined region (Europe, Asia, United States)

## Our Agentic System Architecture:

We will create and use two agents:


**1. web_news_searcher:**

The agent is provided with precise instructions to retrieve financial news from Reuters sources, ensuring that only recent dates are considered. In addition, it is equipped with the *WebSearchTool* to enable web access.

**2. new_evaluator**:

The agent is explicitly instructed to verify three elements:
1. The source link,
2. The publication date, and
3. The user-specified region.

Furthermore, the required output format is defined using a a Python dataclass consisting of two fields: (feedback, score).

The score plays a critical role:

* If all required elements are present in the output of the first agent, the score is marked as “successful.”

* Conversely, if any element is missing, the score is set to “unsuccessful,” in which case the agentic system continues its iterative process.


In [1]:
from IPython.display import Image, display
image_path = path+"/web_news_searcher_BOT.png"
display(Image(filename=image_path))

NameError: name 'path' is not defined

## Install Lib

In order to use OpenAI Agent SDK:

In [2]:
!pip install openai -q

In [1]:
import asyncio
!pip install openai==1.76.0 openai-agents


In [2]:
import openai
print(openai.__version__)

1.76.0


In [4]:
!pip install openai-agents -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.4/169.4 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/365.7 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 26.2 MB/s eta 0:00:00


You need to create an OpenAI API Key:

In [17]:
from google.colab import userdata
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

import os
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [18]:
from datetime import datetime, timedelta

In [19]:
import nest_asyncio
nest_asyncio.apply()

from agents import Agent, Runner, WebSearchTool, trace, ItemHelpers, TResponseInputItem
from dataclasses import dataclass
from typing import Literal

In [20]:
import asyncio

# Evaluator-Optimizer Multi-Agent System (LLM-as-a-Judge)

Defining the two agents: **web_news_searcher** and **news_evaluator**.

In [7]:
today_date = datetime.now().strftime("%Y-%m-%d")
yesterday_date = (datetime.now() - timedelta(1)).strftime("%Y-%m-%d")
two_days_ago = (datetime.now() - timedelta(2)).strftime("%Y-%m-%d")

INSTRUCTIONS_NEWS_SEARCH = (
    "You are a research news assistant specialized in Finance or any news that can have an impact on the financial health of a given company or a region. "
    "You search the web for a given region to collect the most important and recent news. "
    "Produce a concise summary of the results with citations. The summary must be 2-3 paragraphs and less than 300 words. "
    "Capture the main points. Write succinctly."
    f"Get only the latest news between 2 days ago {two_days_ago} and today {today_date}. "
    "Use only this resource: https://www.reuters.com/. "
    "For each item you are providing, specify the date on which the article was published. "
    "For each item you are providing, specify the complete external link to the article. "
    "This is important. Otherwise, don't give me a summary, if you don't provide a date and an external link. "
    "If you don't give a source link, it means that you are hallucinating. "
)

web_news_searcher = Agent(
        name="web_news_searcher",
        instructions=INSTRUCTIONS_NEWS_SEARCH,
        tools=[WebSearchTool()],
    )

@dataclass
class EvaluationFeedback:
    feedback: str
    score: Literal["successful", "unsuccessful"]

news_evaluator = Agent(
    name="news_evaluator",
    instructions=(
        "You will be given a financial news summary in a given region (Euro, US, Asia) and you are asked to evaluate the completeness of external source links. "
        "Check whether the news summary is supported by external article links from https://www.reuters.com/. "
        f"Check whether the news summary was published between 2 days ago {two_days_ago} and today {today_date}. "
        "For each item of news, the date of the news must be explicitly provided."
        "Check whether the news summary corresponds to the region or regions requested by the user. "
        "If the user requested 'n' news items (it could be 4 or 5 or 10 whatever number the user requests), each item must be supported by an external source link, so you need to ensure that there are 'n' source links. "
        "External links are required for the user to verify the accuracy of the news. "

    ),
    output_type=EvaluationFeedback,
)

In [30]:
from datetime import datetime, timedelta
from dataclasses import dataclass
from typing import Literal

from agents import Agent, WebSearchTool, ModelSettings


# ---------------------------------------------------------
# DATE RANGE
# ---------------------------------------------------------

today_date = datetime.now().strftime("%Y-%m-%d")
two_days_ago = (datetime.now() - timedelta(days=2)).strftime("%Y-%m-%d")


# ---------------------------------------------------------
# NEWS SEARCHER INSTRUCTIONS
# ---------------------------------------------------------

INSTRUCTIONS_NEWS_SEARCH = f"""
You are a financial news research assistant.

You MUST use web search.

For every search, explicitly search Reuters using queries equivalent to:

site:reuters.com <topic keywords> <region keywords> <date>

For the current task, search for real Reuters articles published between
{two_days_ago} and {today_date}.

Do not stop after one failed query.

If the first search returns insufficient results:
- reformulate the query
- broaden synonyms
- search by relevant companies
- search by subtopics
- continue until you have exhausted reasonable Reuters searches

Examples for AI business:
- site:reuters.com artificial intelligence business United States
- site:reuters.com OpenAI United States Reuters
- site:reuters.com Nvidia AI business Reuters
- site:reuters.com AI startup funding Reuters
- site:reuters.com AI cloud data centers Reuters
- site:reuters.com AI regulation United States Reuters

Return exactly the number of news items requested by the user.

For every item provide:
1. Headline
2. Publication date
3. Short summary
4. Publisher: Reuters
5. Complete direct Reuters article URL

IMPORTANT:
Do not say that no Reuters articles exist until you have tried multiple
different Reuters-focused searches.

Do not use non-Reuters sources.
Do not fabricate URLs.
Do not use ETF prices or generic market data.
"""

# ---------------------------------------------------------
# NEWS SEARCHER AGENT
# ---------------------------------------------------------

web_news_searcher = Agent(
    name="web_news_searcher",
    instructions=INSTRUCTIONS_NEWS_SEARCH,
    tools=[WebSearchTool()],
    model_settings=ModelSettings(
        tool_choice="required",
    )
)


# ---------------------------------------------------------
# EVALUATION OUTPUT
# ---------------------------------------------------------

@dataclass
class EvaluationFeedback:
    feedback: str
    score: Literal["successful", "unsuccessful"]


# ---------------------------------------------------------
# NEWS EVALUATOR
# ---------------------------------------------------------

INSTRUCTIONS_NEWS_EVALUATOR = f"""
You are a strict evaluator of financial news search results.

You will receive a news summary generated for a user request.

Evaluate whether the answer satisfies ALL of the following requirements:

1. The answer contains exactly the number of news items requested
   by the user.

2. Every item corresponds to a genuine Reuters news article.

3. Every item contains:
   - headline
   - publication date
   - publisher
   - complete external Reuters URL

4. Every publication date must fall between:
   - {two_days_ago}
   - {today_date}

5. Every item must be relevant to the geographical region requested
   by the user.

6. The items must be actual news articles.
   ETF prices, stock quotes, market quote pages, index values,
   and generic market commentary do NOT count as news items.

7. Every article must contain a Reuters external URL.
   If even one required URL is missing, mark the result unsuccessful.

8. Do not assume a URL exists just because Reuters is mentioned.
   The external article link must actually appear in the answer.

SCORING:

- Return "successful" only if ALL requirements are satisfied.
- Otherwise return "unsuccessful".

When unsuccessful, provide specific feedback explaining exactly what
is missing or incorrect so the search agent can perform a new search.
"""


news_evaluator = Agent(
    name="news_evaluator",
    instructions=INSTRUCTIONS_NEWS_EVALUATOR,
    output_type=EvaluationFeedback,
)


In [31]:
news_evaluator

Agent(name='news_evaluator', instructions='\nYou are a strict evaluator of financial news search results.\n\nYou will receive a news summary generated for a user request.\n\nEvaluate whether the answer satisfies ALL of the following requirements:\n\n1. The answer contains exactly the number of news items requested\n   by the user.\n\n2. Every item corresponds to a genuine Reuters news article.\n\n3. Every item contains:\n   - headline\n   - publication date\n   - publisher\n   - complete external Reuters URL\n\n4. Every publication date must fall between:\n   - 2026-09-09\n   - 2026-09-11\n\n5. Every item must be relevant to the geographical region requested\n   by the user.\n\n6. The items must be actual news articles.\n   ETF prices, stock quotes, market quote pages, index values,\n   and generic market commentary do NOT count as news items.\n\n7. Every article must contain a Reuters external URL.\n   If even one required URL is missing, mark the result unsuccessful.\n\n8. Do not ass

In [32]:
news_evaluator.model_settings

ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, metadata=None, store=None, include_usage=None, extra_query=None, extra_body=None, extra_headers=None)

# Chat with the Mutli-Agent System

Now we can run the multi-agent system in a feedback loop:

## Give me the latest 5 news in Euro zone, and specify the resource for each one of them.

In the following code snippet, we create a feedback loop where we start with the execution of the `web_news_searcher` agent in response to a user query:

> Provide the latest 5 news in the Eurozone and specify the source for each one.

The output of this agent, `latest_outline`, is then used as the input for the evaluator agent `news_evaluator`.

We then retrieve the evaluation produced by this agent and store it in a variable: `EvaluationFeedback`.

To avoid the risk of infinite iterative loops, a termination parameter max_iteration = 4 is defined. Under this constraint, the system stops execution in under either of the following two scenarios:
* when a satisfactory response to the user’s query is produced (convergence criterion): *score = successful*
* when the maximum number of iterations (4) is reached (termination criterion).






In [26]:
async def main() -> None:
    msg = input("User's request:")
    input_items: list[TResponseInputItem] = [{"content": msg, "role": "user"}]

    latest_outline: str | None = None

    max_iteration = 4

    nbr_iterations = 0
    while True:
        news_searcher_result = await Runner.run(
            web_news_searcher,
            input_items,
          )

        input_items = news_searcher_result.to_input_list()
        latest_outline = ItemHelpers.text_message_outputs(news_searcher_result.new_items)

        print("\033[92m**************************NEWS SEARCH:**************************\033[0m")
        print(latest_outline)

        print("\n\n\033[92m**************************Running Evaluation:**************************\033[0m")
        news_evaluator_result = await Runner.run(news_evaluator, latest_outline)
        result: EvaluationFeedback = news_evaluator_result.final_output

        print(f"\033[94mEvaluator score: {result.score}\033[0m")
        print("\n")

        print(f"\033[94mEvaluator feedback: {result.feedback}\033[0m")
        print("\n")

        if result.score == "successful":
              print("\033[92m Links check is good ==> stopping the iteration.\033[0m")
              break

        print("\n\n\033[92m**************************Running Refinement:**************************\033[0m")
        print("\033[94mRe-running with feedback for improvements:\033[0m")
        input_items.append({"content": f"News Summary: {latest_outline}", "role": "system"})
        input_items.append({"content": f"Feedback: {result.feedback}\n for each item in the news summary, provide an external source link:", "role": "user"})
        nbr_iterations+=1

        if nbr_iterations >= max_iteration:
              print("\033[91mReached max_iteration ==> stopping the iteration.\033[0m")
              break

    print("\n\n\033[92m**************************Final Set of News with Links:**************************\033[0m")
    print(latest_outline)

if __name__ == "__main__":
    asyncio.run(main())

# Input ==> Give me the latest 5 news items in the Eurozone, and specify the source for each one.

User's request:dame la ultimas noticias interesantes
**************************NEWS SEARCH:**************************
A continuación, se presentan las noticias financieras más recientes y relevantes en Estados Unidos, publicadas entre el 9 y el 11 de septiembre de 2026:

1. **La inflación en Estados Unidos se mantiene en el 3,4% en agosto y deja un camino incierto para la Reserva Federal**
   - **Fecha de publicación:** 11 de septiembre de 2026
   - **Resumen:** En agosto, la inflación en EE.UU. se mantuvo en el 3,4%, igual que en julio, según la Oficina de Estadísticas Laborales. Aunque por debajo del 4,2% de mayo, sigue lejos del objetivo del 2% de la Reserva Federal. La inflación subyacente bajó al 2,4%. Este contexto complica la decisión de la Fed sobre posibles aumentos de los tipos de interés en su próxima reunión, especialmente debido al reciente incremento en los precios energéticos.
   - **Fuente:** ([elpais.com](https://elpais.com/economia/2026-09-11/la-inflacion-en-estados-u

In [33]:
async def main() -> None:
    msg = input("User's request: ")

    input_items: list[TResponseInputItem] = [
        {"content": msg, "role": "user"}
    ]

    latest_outline: str | None = None

    max_iteration = 4
    nbr_iterations = 0

    while True:

        # 1. NEWS SEARCHER
        news_searcher_result = await Runner.run(
            web_news_searcher,
            input_items,
        )

        input_items = news_searcher_result.to_input_list()

        latest_outline = ItemHelpers.text_message_outputs(
            news_searcher_result.new_items
        )

        print(
            "\033[92m"
            "**************************NEWS SEARCH:"
            "**************************"
            "\033[0m"
        )
        print(latest_outline)

        # 2. EVALUATOR
        print(
            "\n\n\033[92m"
            "**************************Running Evaluation:"
            "**************************"
            "\033[0m"
        )

        news_evaluator_result = await Runner.run(
            news_evaluator,
            latest_outline
        )

        result: EvaluationFeedback = (
            news_evaluator_result.final_output
        )

        print(
            f"\033[94mEvaluator score: {result.score}\033[0m"
        )
        print()

        print(
            f"\033[94mEvaluator feedback: {result.feedback}\033[0m"
        )
        print()

        # 3. SUCCESS?
        if result.score == "successful":
            print(
                "\033[92m"
                "Links check is good ==> stopping the iteration."
                "\033[0m"
            )
            break

        # 4. MAX ITERATIONS?
        nbr_iterations += 1

        if nbr_iterations >= max_iteration:
            print(
                "\033[91m"
                "Reached max_iteration ==> stopping the iteration."
                "\033[0m"
            )
            break

        # 5. REFINEMENT
        print(
            "\n\n\033[92m"
            "**************************Running Refinement:"
            "**************************"
            "\033[0m"
        )

        print(
            "\033[94m"
            "Re-running with feedback for improvements:"
            "\033[0m"
        )

        input_items.append(
            {
                "content": f"News Summary: {latest_outline}",
                "role": "system",
            }
        )

        input_items.append(
            {
                "content": (
                    f"Feedback: {result.feedback}\n"
                    "For each item in the news summary, "
                    "provide an external source link."
                ),
                "role": "user",
            }
        )

    print(
        "\n\n\033[92m"
        "**************************Final Set of News with Links:"
        "**************************"
        "\033[0m"
    )

    print(latest_outline)

In [34]:
await main()

User's request: Give me the latest 5 Reuters articles from the last 2 days about OpenAI, Nvidia, Oracle, Adobe, AI infrastructure or AI investment in the United States.
**************************NEWS SEARCH:**************************
As of September 11, 2026, there are no Reuters articles published in the past two days specifically covering OpenAI, Nvidia, Oracle, Adobe, AI infrastructure, or AI investment in the United States.

However, here are the latest stock prices for the mentioned companies:

- **Nvidia Corp (NVDA)**: $219.02, up $0.66 (0.30%)
- **Oracle Corp (ORCL)**: $151.62, down $1.32 (-0.86%)
- **Adobe Inc (ADBE)**: $252.55, up $3.72 (1.50%)

Please note that stock prices are subject to change and may have fluctuated since this information was retrieved. 


**************************Running Evaluation:**************************
Evaluator score: unsuccessful

Evaluator feedback: The response did not provide any Reuters news articles. Instead, it provided stock prices, which 

As shown in the above output. The `web_news_searcher` provided several news in the first step of the loop (at the time this notebook was executed):

**************************NEWS SEARCH:**************************
Here are the five **most recent** news items related to the Eurozone, as of **September 6, 2025**, along with their sources:

1. **Google hit with a €2.95 billion EU antitrust fine**  
   The European Commission fined Alphabet’s Google a record-breaking **€2.95 billion** (~$3.45 billion) for antitrust violations in ad‑tech.  
   Source: Reuters, published on **September 5, 2025** ([reuters.com](https://www.reuters.com/legal/litigation/google-hit-with-345-billion-eu-antitrust-fine-over-adtech-practices-2025-09-05/?utm_source=openai))

2. **Eurozone’s GDP confirmed at a modest 0.1% growth in Q2**  
   Final figures from Eurostat reaffirmed that the Eurozone economy grew by **0.1% quarter-on-quarter** in the second quarter of 2025.  
   Source: Eurostat announcement (via RTT News) and confirmed by additional outlets ([rttnews.com](https://www.rttnews.com/3571989/eurozone-gdp-growth-confirmed-at-0-1.aspx?utm_source=openai), [ec.europa.eu](https://ec.europa.eu/eurostat/web/products-euro-indicators/w/2-05092025-ap?utm_source=openai))

3. **Germany’s 2025 growth forecast downgraded with rebound expected later**  
   The German Institute for Economic Research (DIW Berlin) projected a subdued **0.2% GDP growth** for Germany in 2025 (down from 0.3%), with stronger rebounds expected in **2026 (1.7 %)** and **2027 (1.8 %)**. Persistent structural issues like low productivity, labor shortages, and high energy costs remain concerns.  
   Source: Reuters analysis on **September 5, 2025** ([reuters.com](https://www.reuters.com/markets/europe/germany-set-tepid-2025-growth-accelerating-2026-2027-diw-says-2025-09-05/?utm_source=openai))

4. **Eurozone economy expanded at its slowest pace since late 2023**  
   The Q2 2025 GDP growth of **0.1%** was identified as the weakest since Q4 2023. Household spending was sluggish; capital formation and exports declined, although public spending and inventories provided support.  
   Source: TradingEconomics summary of Eurostat data on **September 5, 2025** ([tradingeconomics.com](https://tradingeconomics.com/euro-area/gdp-growth/news/483190?utm_source=openai))

5. **Global markets themes: Eurozone watchers expect steady rates amid ECB caution**  
   In the “Take Five: Winds of Change” briefing, Reuters noted that markets expect the ECB to **hold interest rates steady** in the near term, with attention focused on key economic releases and speeches, including those from ECB President Lagarde.  
   Source: Reuters, published on **September 5, 2025** ([reuters.com](https://www.reuters.com/business/take-five/global-markets-themes-graphic-2025-09-05/?utm_source=openai))


However the `news_evaluator` agent generated an `unsuccessful` score and provided this feedback, where only 3 news contain direct link to reuters.com:

> Evaluator feedback: The news summary provides five items pertaining to the Eurozone, as requested. Of the five, only three (items 1, 3, and 5) contain direct external source links to reuters.com, meeting the fundamental requirement for authenticating the news via reliable and easily verifiable sources. Items 2 and 4 reference other external sources (Eurostat/RTT News and TradingEconomics), and do not include reuters.com links. All items carry explicit publication dates falling within the accepted time window (September 4–6, 2025). Each news item is appropriately focused on the Eurozone. However, per the requirements, since only three of the five items provide reuters.com links, the summary is unsuccessful in meeting completeness for external source links as specified.







This feedback is sent again to the `web_news_search` to search for only reuters links. The latest produced again another 5 news. However some of them did not respect the time constraint (older than the last 2 days):


> While every news item is appropriately sourced and Eurozone-specific, only 2 of 5 are strictly within the user's requested publication window (2025-09-04 to 2025-09-06). The others are slightly early. Thus, the response is unsuccessful in fully meeting the date criterion for all five news items, despite solid source linkage and correct regional focus.



The feedback is sent again to the `news_web_search` agent, which performs another search and this time satisfies all the constraints. The `news_evaluator` returns a `successful` evaluation.

### Another Run

I ran the same question again to estimate the convergence speed (I get the same speed than the run above):

In [ ]:
if __name__ == "__main__":
    asyncio.run(main())

# Input ==> Give me the latest 5 news items in Eurozone, and specify the source for each one.

User's request:Give me the latest 5 news items in Eurozone, and specify the source for each one.
**************************NEWS SEARCH:**************************
Here are the latest **five** key news items related to the **Eurozone**, as of **September 5–6, 2025**, along with their sources and publication dates:

1. Google hit with €2.95 billion EU antitrust fine over adtech practices  
   — Alphabet’s Google was fined €2.95 billion (~USD 3.45 billion) by the European Commission for anticompetitive behavior in its adtech operations. This decision was announced on **September 5, 2025**, and highlights the EU’s ongoing scrutiny of big tech. ([reuters.com](https://www.reuters.com/legal/litigation/google-hit-with-345-billion-eu-antitrust-fine-over-adtech-practices-2025-09-05/?utm_source=openai))

2. Eurozone GDP and employment up by 0.1% in Q2 2025  
   — Eurostat confirmed that in the **second quarter of 2025**, the euro area’s seasonally adjusted GDP rose by **0.1%**, and employment also

In [ ]:
# # Another kind of outputs that you can have from the bot:
# Below are external **Reuters** links for each of the five Eurozone news items, as requested:

# 1. **Eurozone Q2 GDP growth confirmed at 0.1% (Resilience despite weak environment)**
#    – Report: euro zone economy grew by 0.1% in Q2 2025, aided by Spain, France, and Ireland, offsetting contraction in Germany and Italy.
#    – Source: “Euro zone growth holds up better than feared in Q2” ([reuters.com](https://www.reuters.com/business/euro-zone-growth-holds-up-better-than-feared-q2-2025-07-30/?utm_source=openai))

# 2. **August PMI shows sluggish euro‑zone growth**
#    – Report: Eurozone Composite PMI edged up to 51.0 in August from 50.9, signaling continued modest expansion; services slowed while manufacturing strengthened.
#    – Source: “Euro zone economic growth remains sluggish in August, PMI shows” ([reuters.com](https://www.reuters.com/world/europe/euro-zone-economic-growth-remains-sluggish-august-pmi-shows-2025-09-03/?utm_source=openai))

# 3. **Germany inaugurates the “Jupiter” supercomputer (Europe’s first exascale)**
#    – Report: German Chancellor Friedrich Merz inaugurated the Jupiter supercomputer at Jülich research centre; Europe’s first exascale capable of one billion billion calculations per second.
#    – Source: “Germany's Merz inaugurates Nvidia supercomputer for research” ([reuters.com](https://www.reuters.com/business/media-telecom/germanys-merz-inaugurates-nvidia-supercomputer-research-2025-09-05/?utm_source=openai))

# 4. **France’s political crisis imperils credit rating amid looming confidence vote**
#    – Report: France faces a confidence vote on September 8; political turmoil risks a sovereign credit downgrade from Fitch, which could push France into A+ territory.
#    – Source: “France faces sovereign rating test amid political chaos” ([reuters.com](https://www.reuters.com/business/finance/france-faces-sovereign-rating-test-amid-political-chaos-2025-09-05/?utm_source=openai))
#    – Additionally: “What you need to know about France's political crisis” ([reuters.com](https://www.reuters.com/world/europe/what-you-need-know-about-frances-political-crisis-2025-09-05/?utm_source=openai))

# 5. **European factories return to growth**
#    – Report: Eurozone manufacturing expanded in August for the first time since mid‑2022, with PMI rising to 50.7 due to strong domestic demand.
#    – Source: “European factories return to growth, Asia activity shrinks” ([reuters.com](https://www.reuters.com/world/china/global-economy-european-factories-return-growth-asia-activity-shrinks-2025-09-01/?utm_source=openai))


## Give me the latest news for 2 regions: Eurozone and US

Now I'm looking at two regions in the same time:

In [ ]:
if __name__ == "__main__":
    asyncio.run(main())

# Input ==> Give me the latest 5 news items for each region: Eurozone and US. Specify the source for each one, and organize the news into separate paragraphs by region.

User's request:Give me the latest 5 news items for each region: Eurozone and US. Specify the source for each one, and organize the news into separate paragraphs by region.
**************************NEWS SEARCH:**************************
Here are the **five latest news items** for each region—**Eurozone** and **United States**—organized into separate paragraphs by region. Each item includes its source and precise date where available to avoid ambiguity.

---

Eurozone

1. Eurostat confirmed that the eurozone economy grew by 0.1% in Q2 2025, with the EU overall up by 0.2%. Denmark led regional growth while Germany and Italy lagged behind ([uk.finance.yahoo.com](https://uk.finance.yahoo.com/news/eurozone-gdp-employment-rise-113549218.html?utm_source=openai)).
2. RTT News echoed that the eurozone registered marginal growth of 0.1% in the second quarter, noting that higher consumption partly offset declines elsewhere ([rttnews.com](https://www.rttnews.com/3571989/eurozone-gdp-growth-confirm

As you can see, the news_web_search agent did provide the news for both regions and the evaluation, news_evaluator agent, was performed also on both regions.

### Another Run

To evaluate the convergence speed, we repeated the process again.

In most cases, the target output is obtained within one to three loops.

In [ ]:
if __name__ == "__main__":
    asyncio.run(main())

# Input ==> Give me the latest 5 news items for each region: Eurozone and US. Specify the source for each one, and organize the news into separate paragraphs by region.

User's request:Give me the latest 5 news items for each region: Eurozone and US. Specify the source for each one, and organize the news into separate paragraphs by region.
**************************NEWS SEARCH:**************************
Here are the five **most recent news items** for each region—**Eurozone** and the **United States**—as of **September 6, 2025**. Each item includes its source and is organized by region with distinct paragraphs.

---

Eurozone:

1. Europe’s economy expanded by just **0.1% in Q2 2025**, marking a sluggish performance that lags far behind the U.S. growth rate. German contraction and ongoing trade uncertainties contributed to this subdued outcome. ([courthousenews.com](https://www.courthousenews.com/europes-economy-flatlines-at-0-1-as-us-races-8-times-faster/?utm_source=openai), [uk.finance.yahoo.com](https://uk.finance.yahoo.com/news/eurozone-gdp-employment-rise-113549218.html?utm_source=openai))

2. Final data confirmed the marginal **0.1% GDP growth** i